# 98 — Kiểm thực nghiệm repo gốc CausalGym (điểm 8, 9, 10 của sổ §8)

Notebook này **không thay** việc đọc code. Thứ tự đúng cho TV-B:

1. Đọc code theo `docs/PHIEU_KIEM_REPO_GOC.md`, ghi "khớp/khác" cho 12 điểm **trước**.
2. Sau đó mới chạy notebook này để có số liệu thực nghiệm cho điểm 8, 9, 10.

Notebook chỉ chạy CPU, vài phút. Phần pyvene (ô 6) cần môi trường cũ của repo gốc
(`transformers==4.46.2`, `pyvene==0.1.6`), nên mặc định tắt khi chạy trong môi trường dự án.

Kết quả ghi dần ra `OUT_DIR/kiem_repo_goc.jsonl` (QT-23).

In [1]:
# Tham số (papermill). Comment trong ô này không có dấu bằng (QT-21).
REPO_URL = "https://github.com/aryamanarora/causalgym.git"
REPO_COMMIT = "0f3129f"
REPO_DIR = "causalgym_goc"
# Tên tokenizer trên HF, hoặc đường dẫn tới file tokenizer.json của GPT-NeoX-20B
TOKENIZER = "EleutherAI/pythia-70m"
TASKS = ["agr_sv_num_subj-relc", "npi_any_subj-relc", "filler_gap_subj", "cleft"]
# So với dataset HF aryaman/causalgym; cần Internet
COMPARE_HF = True
HF_DATASET = "aryaman/causalgym"
# Ô 6 cần transformers 4.46.2 và pyvene 0.1.6
RUN_PYVENE_TEST = False
OUT_DIR = "outputs/98_kiem_repo_goc"
# Chế độ thử trên CPU, không Internet: dùng repo đã clone sẵn, không so HF
TOY_TEST = False

In [2]:
# Parameters
REPO_DIR = "/home/claude/ext/causalgym"
TOKENIZER = "/home/claude/ext/tok/BlinkDL_RWKV-LM_main_RWKV-v4_20B_tokenizer.json"
COMPARE_HF = False
RUN_PYVENE_TEST = True
TOY_TEST = True
OUT_DIR = "/home/claude/work/runs/out98"


## Ô 1. Phiên bản thư viện (QT-26)

In [3]:
import os, sys, json, hashlib, subprocess, platform, importlib
import torch, transformers
print("python", platform.python_version(), "| torch", torch.__version__, "| transformers", transformers.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "không (CPU)")
os.makedirs(OUT_DIR, exist_ok=True)
OUT_JSONL = os.path.join(OUT_DIR, "kiem_repo_goc.jsonl")
open(OUT_JSONL, "w").close()

def ghi(rec):
    with open(OUT_JSONL, "a") as f:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

python 3.13.18 | torch 2.15.0+cu132 | transformers 4.46.2
GPU: không (CPU)


## Ô 2. Lấy repo gốc đúng commit và tokenizer

Clone vào `REPO_DIR` rồi `git checkout REPO_COMMIT`. Chỉ import `data.py`, `interventions.py`
(không import `das.py` vì file đó xoá thư mục `figs/das/steps` khi import).
`data.py` gọi `random.seed(42)` khi import, giống lúc tác giả chạy.

In [4]:
REPO_DIR = os.path.abspath(REPO_DIR)
if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "clone", "-q", REPO_URL, REPO_DIR], check=True)
subprocess.run(["git", "-C", REPO_DIR, "checkout", "-q", REPO_COMMIT], check=True)
head = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
print("repo HEAD:", head)
assert head.startswith(REPO_COMMIT), "Sai commit"

from transformers import AutoTokenizer, PreTrainedTokenizerFast
if TOKENIZER.endswith(".json"):
    tok = PreTrainedTokenizerFast(tokenizer_file=TOKENIZER, eos_token="<|endoftext|>", bos_token="<|endoftext|>", unk_token="<|endoftext|>")
else:
    tok = AutoTokenizer.from_pretrained(TOKENIZER)
tok.pad_token = tok.eos_token
print("tokenizer:", TOKENIZER, "| vocab", len(tok), "| eos id", tok.eos_token_id)

sys.path.insert(0, REPO_DIR)
_cwd = os.getcwd()
os.chdir(REPO_DIR)            # data.py đọc data/templates theo đường dẫn tương đối
import data as cg_data
importlib.reload(cg_data)     # đảm bảo random.seed(42) của data.py chạy lại
os.chdir(_cwd)
ghi({"buoc": "repo", "commit": head, "tokenizer": TOKENIZER, "vocab": len(tok)})

repo HEAD: 0f3129ff3b6c5c8264892f30a25be25150ae9179


tokenizer: /home/claude/ext/tok/BlinkDL_RWKV-LM_main_RWKV-v4_20B_tokenizer.json | vocab 50277 | eos id 0


## Ô 3. Điểm 10: tập eval của bài = split `test` của HF?

Sinh lại theo **hai đường code** của repo:

- (a) `convert_to_huggingface_format` (`data.py:380–417`): train seed 42 (100 batch × 4), dev seed 420, test seed 1 (25 batch × 4), loại trùng theo train.
- (b) `das.py:73–83` với tham số của `test_all.py:24–31` (`steps=100`, `batch_size=4`, `eval_seed=1` khi không bật `hparam_non_das`).

Rồi so (a) với (b), và nếu có Internet thì so với split `test` và `validation` trên HF.
Kết luận "khớp" chỉ khi cả base, src, base_label, src_label trùng **theo đúng thứ tự** cho cả 100 dòng.

In [5]:
def pairs_to_rows(batches, task):
    return [{"base": p.base, "src": p.src, "base_type": p.base_type, "src_type": p.src_type,
             "base_label": p.base_label, "src_label": p.src_label, "task": task}
            for b in batches for p in b.pairs]

def h(rows):
    return hashlib.sha256(json.dumps(rows, sort_keys=True).encode()).hexdigest()[:16]

os.chdir(REPO_DIR)
REGEN = {}
for task in TASKS:
    name = f"syntaxgym/{task}"
    # (a) đường xuất HF
    ds = cg_data.Dataset.load_from(name)
    train = ds.sample_batches(tok, 4, 100, "cpu", seed=42, manipulate=None)
    discard = {''.join(p.base) for b in train for p in b.pairs}
    dev = ds.sample_batches(tok, 4, 25, "cpu", seed=420, manipulate=None, discard=discard)
    test = ds.sample_batches(tok, 4, 25, "cpu", seed=1, manipulate=None, discard=discard)
    # (b) đường das.py + test_all.py
    ds2 = cg_data.Dataset.load_from(name)
    train2 = ds2.sample_batches(tok, 4, 100, "cpu", seed=42, manipulate=None)
    discard2 = set()
    for b in train2:
        for p in b.pairs:
            discard2.add(''.join(p.base))
    evalset = ds2.sample_batches(tok, 4, 25, "cpu", seed=1, discard=discard2, manipulate=None)
    ra, rb = pairs_to_rows(test, task), pairs_to_rows(evalset, task)
    REGEN[task] = {"train": pairs_to_rows(train, task), "dev": pairs_to_rows(dev, task),
                   "test": ra, "evalset_batches": evalset, "discard": discard, "ds": ds2}
    # Mỗi batch 4 = 2 cặp lấy mẫu (data.py:302-312) + 2 cặp đảo (data.py:334-335)
    sampled = [p for b in test for p in b.pairs[:2]]
    rec = {"buoc": "diem10_sinh_lai", "task": task, "n_test": len(ra), "a_bang_b": ra == rb,
           "hash_test": h(ra), "hash_dev": h(REGEN[task]["dev"]),
           "so_cau_khac_nhau_trong_train": len(discard),
           "base_cap_lay_mau_trong_train": sum(''.join(p.base) in discard for p in sampled),
           "src_cap_lay_mau_trong_train": sum(''.join(p.src) in discard for p in sampled),
           "ty_le_cau_base_eval_trong_train": round(sum(''.join(r["base"]) in discard for r in ra) / len(ra), 3)}
    print(rec); ghi(rec)
os.chdir(_cwd)

{'buoc': 'diem10_sinh_lai', 'task': 'agr_sv_num_subj-relc', 'n_test': 100, 'a_bang_b': True, 'hash_test': '728e3aa603debb27', 'hash_dev': 'dfce1b2af2d85d6e', 'so_cau_khac_nhau_trong_train': 395, 'base_cap_lay_mau_trong_train': 0, 'src_cap_lay_mau_trong_train': 4, 'ty_le_cau_base_eval_trong_train': 0.04}


{'buoc': 'diem10_sinh_lai', 'task': 'npi_any_subj-relc', 'n_test': 100, 'a_bang_b': True, 'hash_test': 'e3f559528d49bed2', 'hash_dev': '7c5e3c5e851b4260', 'so_cau_khac_nhau_trong_train': 400, 'base_cap_lay_mau_trong_train': 0, 'src_cap_lay_mau_trong_train': 0, 'ty_le_cau_base_eval_trong_train': 0.0}


{'buoc': 'diem10_sinh_lai', 'task': 'filler_gap_subj', 'n_test': 100, 'a_bang_b': True, 'hash_test': 'c31e363ad4a7ab64', 'hash_dev': '30e0c6f0decc1c4f', 'so_cau_khac_nhau_trong_train': 394, 'base_cap_lay_mau_trong_train': 0, 'src_cap_lay_mau_trong_train': 0, 'ty_le_cau_base_eval_trong_train': 0.0}


{'buoc': 'diem10_sinh_lai', 'task': 'cleft', 'n_test': 100, 'a_bang_b': True, 'hash_test': '733fd80e3b71a848', 'hash_dev': '05f7ac5e6854eaaf', 'so_cau_khac_nhau_trong_train': 196, 'base_cap_lay_mau_trong_train': 0, 'src_cap_lay_mau_trong_train': 50, 'ty_le_cau_base_eval_trong_train': 0.5}


In [6]:
# So với HF (cần Internet). Bỏ qua khi TOY_TEST hoặc COMPARE_HF tắt.
if COMPARE_HF and not TOY_TEST:
    from datasets import load_dataset
    hf = load_dataset(HF_DATASET)
    print({k: len(v) for k, v in hf.items()})
    KEYS = ["base", "src", "base_type", "src_type", "base_label", "src_label"]
    for task in TASKS:
        out = {"buoc": "diem10_so_hf", "task": task}
        # test HF so với seed 1; validation HF so với seed 1 (phải khác) và seed 420 (phải trùng)
        for key, hf_split, mine in [("test_vs_seed1", "test", REGEN[task]["test"]),
                                    ("validation_vs_seed1", "validation", REGEN[task]["test"]),
                                    ("validation_vs_seed420", "validation", REGEN[task]["dev"])]:
            rows = [r for r in hf[hf_split] if r["task"] == task]
            same = [all(r[k] == m[k] for k in KEYS) for r, m in zip(rows, mine)]
            out[f"{key}_n_hf"] = len(rows)
            out[f"{key}_n_trung"] = int(sum(same))
        n = len(REGEN[task]["test"])
        out["ket_luan"] = ("test HF = tập eval seed 1 của bài"
                           if out["test_vs_seed1_n_trung"] == out["test_vs_seed1_n_hf"] == n else "KHÁC, xem chi tiết")
        print(out); ghi(out)
else:
    print("Bỏ qua so HF (TOY_TEST hoặc COMPARE_HF = False)")

Bỏ qua so HF (TOY_TEST hoặc COMPARE_HF = False)


## Ô 4. Điểm 9: lỗi đảo vị trí ảnh hưởng bao nhiêu cặp, ở vùng nào?

Dùng chính `Batch.compute_pos("last")` của repo (`data.py:115–142`). Hàm trả `[vị trí src, vị trí base]`.
Lời gọi pyvene (`eval.py:27–31`, `train.py:58–62`) lấy source ở `pos_interv[1]` (vị trí **base**).
Lỗi chỉ có tác dụng ở vùng mà vị trí base ≠ vị trí src, tức khi base và src khác số token ở một vùng đứng trước hoặc tại vùng đó.

Bảng in ra: với mỗi tác vụ và mỗi vùng mà repo can thiệp (từ `first_var_pos`), tỉ lệ cặp có vị trí lệch.

In [7]:
import collections
for task in TASKS:
    ds = REGEN[task]["ds"]
    counts = collections.Counter(); n = 0; any_diff = 0
    lens_base = collections.Counter()
    for b in REGEN[task]["evalset_batches"]:
        pos_src, pos_base = b.compute_pos("last")
        for i in range(len(b.pairs)):
            n += 1; d_any = False
            for span_i in range(ds.first_var_pos, ds.length):
                ps, pb = pos_src[i][span_i], pos_base[i][span_i]
                if ps != pb:
                    counts[span_i] += 1; d_any = True
            any_diff += d_any
    per_region = {f"{s}:{ds.span_names[s]}": round(counts[s] / n, 3) for s in range(ds.first_var_pos, ds.length)}
    rec = {"buoc": "diem9_ty_le_lech", "task": task, "n_cap": n, "ty_le_cap_co_lech": round(any_diff / n, 3),
           "ty_le_lech_theo_vung": per_region}
    print(json.dumps(rec, ensure_ascii=False)); ghi(rec)

{"buoc": "diem9_ty_le_lech", "task": "agr_sv_num_subj-relc", "n_cap": 100, "ty_le_cap_co_lech": 0.16, "ty_le_lech_theo_vung": {"1:{subject}": 0.16, "2:_that": 0.16, "3:{verbed}": 0.16, "4:_the": 0.16, "5:{object}": 0.16}}
{"buoc": "diem9_ty_le_lech", "task": "npi_any_subj-relc", "n_cap": 100, "ty_le_cap_co_lech": 0.0, "ty_le_lech_theo_vung": {"1:{det1}": 0.0, "2:{np}": 0.0, "3:_that": 0.0, "4:{rc_verb}": 0.0, "5:{det2}": 0.0, "6:{rc_obj}": 0.0, "7:_has": 0.0, "8:{matrix_v}": 0.0}}
{"buoc": "diem9_ty_le_lech", "task": "filler_gap_subj", "n_cap": 100, "ty_le_cap_co_lech": 0.0, "ty_le_lech_theo_vung": {"2:{gap}": 0.0, "3:_the": 0.0, "4:{np1}": 0.0, "5:{clause.verb}": 0.0, "6:_the": 0.0, "7:{clause.obj}": 0.0, "8:{clause.pp}": 0.0}}
{"buoc": "diem9_ty_le_lech", "task": "cleft", "n_cap": 100, "ty_le_cap_co_lech": 0.08, "ty_le_lech_theo_vung": {"2:{verb}": 0.08, "3:_was": 0.08}}


## Ô 5. Điểm 8: căn vùng bằng token hoá từng vùng có khớp token hoá cả câu?

Repo tính vị trí bằng `tokenizer.tokenize(vùng)` cho từng vùng (`data.py:49–60`) nhưng đưa vào model
`tokenizer(cả câu)` (`data.py:63–64`). Nếu BPE gộp token qua ranh giới vùng, vị trí sẽ lệch. Đếm số câu lệch.

In [8]:
for task in TASKS:
    bad = 0; tot = 0
    for r in REGEN[task]["test"]:
        for side in ("base", "src"):
            spans = r[side]
            per_span = []
            for s in spans:
                per_span += tok.convert_tokens_to_ids(tok.tokenize(s))
            full = tok(''.join(spans))["input_ids"]
            tot += 1; bad += int(per_span != full)
    rec = {"buoc": "diem8_tok_mismatch", "task": task, "n_cau": tot, "n_lech": bad}
    print(rec); ghi(rec)

{'buoc': 'diem8_tok_mismatch', 'task': 'agr_sv_num_subj-relc', 'n_cau': 200, 'n_lech': 0}
{'buoc': 'diem8_tok_mismatch', 'task': 'npi_any_subj-relc', 'n_cau': 200, 'n_lech': 0}
{'buoc': 'diem8_tok_mismatch', 'task': 'filler_gap_subj', 'n_cau': 200, 'n_lech': 0}
{'buoc': 'diem8_tok_mismatch', 'task': 'cleft', 'n_cau': 200, 'n_lech': 0}


## Ô 6. Điểm 9 bằng thực nghiệm: pyvene lấy source ở vị trí nào?

Chỉ chạy khi `RUN_PYVENE_TEST = True` trong môi trường cũ (`pip install "transformers==4.46.2" "pyvene==0.1.6"`).
Dựng một GPT-NeoX ngẫu nhiên 2 layer (không cần tải trọng số), một cặp có base và src **khác số token**.
Can thiệp vanilla ở layer 0, `block_output`, gọi **y như** `eval.py:27–31`.
Làm chuẩn bằng tay: tự vá đầu ra layer 0 của base tại vị trí can thiệp bằng trạng thái của source ở
(i) vị trí base, (ii) vị trí src, rồi lấy logits tại vị trí đó. So logits pyvene trả về với hai chuẩn này.
Nếu khớp (i) mà không khớp (ii), lỗi đảo vị trí được xác nhận bằng chạy thật, không chỉ bằng đọc code.
(Lưu ý: hook đặt trên layer không bắt được lượt forward bên trong pyvene, nên phép thử so ở logits.)
Sau đó gọi lại với vị trí source đã sửa (`pos_interv[0]`) để thấy kết quả chuyển sang khớp (ii).

In [9]:
if RUN_PYVENE_TEST:
    import pyvene as pv
    from transformers import GPTNeoXConfig, GPTNeoXForCausalLM
    sys.path.insert(0, REPO_DIR)
    import interventions as cg_int
    torch.manual_seed(0)
    cfg = GPTNeoXConfig(vocab_size=len(tok), hidden_size=64, num_hidden_layers=2, num_attention_heads=4,
                        intermediate_size=128, max_position_embeddings=128)
    model = GPTNeoXForCausalLM(cfg).eval()

    # Cặp tự dựng: vùng chủ ngữ của src dài hơn base; vùng động từ (vùng 3) bị lệch vị trí
    base_spans = ["<|endoftext|>The", " cat", " that", " is"]
    src_spans = ["<|endoftext|>The", " hippopotamuses", " that", " is"]
    print([tok.tokenize(s) for s in base_spans]); print([tok.tokenize(s) for s in src_spans])
    pair = cg_data.Pair(base_spans, src_spans, "singular", "plural", " is", " are")
    batch = cg_data.Batch([pair], tok, "cpu")
    pos_src, pos_base = batch.compute_pos("last")
    REGION = 2                      # vùng " that", đứng sau vùng lệch độ dài
    p_b, p_s = pos_base[0][REGION][0], pos_src[0][REGION][0]
    print("vị trí base", p_b, "| vị trí src", p_s)
    assert p_b != p_s

    cfgi = cg_int.intervention_config("block_output", pv.VanillaIntervention, 0, 0)

    with torch.no_grad():
        src_h = model(**batch.src, output_hidden_states=True).hidden_states[1][0]   # đầu ra layer 0 của source

    def va_tay(vec):
        # Chuẩn: tự vá đầu ra layer 0 của base tại p_b bằng vec, lấy logits tại p_b
        def _hook(m, args, out):
            hs = out[0].clone(); hs[0, p_b] = vec
            return (hs,) + tuple(out[1:])
        hk = model.gpt_neox.layers[0].register_forward_hook(_hook)
        with torch.no_grad():
            lg = model(**batch.base).logits[0, p_b]
        hk.remove()
        return lg

    ref_p_base = va_tay(src_h[p_b])   # nếu đọc source ở vị trí base
    ref_p_src = va_tay(src_h[p_s])    # nếu đọc source ở vị trí src
    assert not torch.allclose(ref_p_base, ref_p_src), "Hai giả thuyết cho cùng logits, phép thử vô hiệu"

    def chay(source_pos):
        iv = pv.IntervenableModel(cfgi, model); iv.disable_model_gradients()
        pos_interv = [[x[REGION] for x in y] for y in batch.compute_pos("last")]
        with torch.no_grad():
            _, cf = iv(batch.base, [None, batch.src], {"sources->base": ([None, source_pos(pos_interv)], pos_interv)},
                       output_original_output=True)
        return cf.logits[0, p_b]

    ra = chay(lambda P: P[1])     # đúng như repo gốc
    rb = chay(lambda P: P[0])     # đã sửa: lấy source ở vị trí của source
    res = {"buoc": "diem9_pyvene", "p_base": p_b, "p_src": p_s,
           "repo_lech_voi_gt_p_base": float((ra - ref_p_base).abs().max()),
           "repo_lech_voi_gt_p_src": float((ra - ref_p_src).abs().max()),
           "sua_lech_voi_gt_p_base": float((rb - ref_p_base).abs().max()),
           "sua_lech_voi_gt_p_src": float((rb - ref_p_src).abs().max())}
    repo_bug = res["repo_lech_voi_gt_p_base"] < 1e-4 < res["repo_lech_voi_gt_p_src"]
    fix_ok = res["sua_lech_voi_gt_p_src"] < 1e-4 < res["sua_lech_voi_gt_p_base"]
    res["ket_luan"] = ("repo đọc source ở vị trí base (lỗi xác nhận); bản sửa đọc đúng vị trí src"
                       if repo_bug and fix_ok else "xem lại: phép thử không phân biệt được")
    print(json.dumps(res, ensure_ascii=False, indent=1)); ghi(res)
else:
    print("Bỏ qua ô pyvene (RUN_PYVENE_TEST = False)")

nnsight is not detected. Please install via 'pip install nnsight' for nnsight backend.
[['<|endoftext|>', 'The'], ['Ġcat'], ['Ġthat'], ['Ġis']]
[['<|endoftext|>', 'The'], ['Ġhipp', 'opot', 'am', 'uses'], ['Ġthat'], ['Ġis']]
vị trí base 3 | vị trí src 6
{
 "buoc": "diem9_pyvene",
 "p_base": 3,
 "p_src": 6,
 "repo_lech_voi_gt_p_base": 0.0,
 "repo_lech_voi_gt_p_src": 0.8061598539352417,
 "sua_lech_voi_gt_p_base": 0.8061598539352417,
 "sua_lech_voi_gt_p_src": 0.0,
 "ket_luan": "repo đọc source ở vị trí base (lỗi xác nhận); bản sửa đọc đúng vị trí src"
}


## Ô 7. Tóm tắt

In lại toàn bộ `kiem_repo_goc.jsonl`. TV-B chép các dòng `ket_luan` vào cột "Chạy thử" của phiếu.

In [10]:
for line in open(OUT_JSONL):
    print(line.rstrip())

{"buoc": "repo", "commit": "0f3129ff3b6c5c8264892f30a25be25150ae9179", "tokenizer": "/home/claude/ext/tok/BlinkDL_RWKV-LM_main_RWKV-v4_20B_tokenizer.json", "vocab": 50277}
{"buoc": "diem10_sinh_lai", "task": "agr_sv_num_subj-relc", "n_test": 100, "a_bang_b": true, "hash_test": "728e3aa603debb27", "hash_dev": "dfce1b2af2d85d6e", "so_cau_khac_nhau_trong_train": 395, "base_cap_lay_mau_trong_train": 0, "src_cap_lay_mau_trong_train": 4, "ty_le_cau_base_eval_trong_train": 0.04}
{"buoc": "diem10_sinh_lai", "task": "npi_any_subj-relc", "n_test": 100, "a_bang_b": true, "hash_test": "e3f559528d49bed2", "hash_dev": "7c5e3c5e851b4260", "so_cau_khac_nhau_trong_train": 400, "base_cap_lay_mau_trong_train": 0, "src_cap_lay_mau_trong_train": 0, "ty_le_cau_base_eval_trong_train": 0.0}
{"buoc": "diem10_sinh_lai", "task": "filler_gap_subj", "n_test": 100, "a_bang_b": true, "hash_test": "c31e363ad4a7ab64", "hash_dev": "30e0c6f0decc1c4f", "so_cau_khac_nhau_trong_train": 394, "base_cap_lay_mau_trong_train": 